In [1]:
import pandas as pd
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler

In [2]:
# Make dataframe
california_housing = fetch_california_housing(as_frame=True)
df = california_housing.frame
df.head()

,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude,MedHouseVal
0,8.3252,41.0,6.984127,1.023810,322.0,2.555556,37.88,-122.23,4.526
1,8.3014,21.0,6.238137,0.971880,2401.0,2.109842,37.86,-122.22,3.585
2,7.2574,52.0,8.288136,1.073446,496.0,2.802260,37.85,-122.24,3.521
3,5.6431,52.0,5.817352,1.073059,558.0,2.547945,37.85,-122.25,3.413
4,3.8462,52.0,6.281853,1.081081,565.0,2.181467,37.85,-122.25,3.422


In [3]:
df.shape

(20640, 9)

In [4]:
# Prepare the data for machine learning
california_data = pd.DataFrame(california_housing.data, columns=california_housing.feature_names)
california_data['MEDV'] = california_housing.target

X = california_data.drop('MEDV', axis=1)
y = california_data['MEDV']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
display(X_train[['AveRooms', 'AveBedrms', 'AveOccup', 'Population']].describe())

,AveRooms,AveBedrms,AveOccup,Population
count,16512.000000,16512.000000,16512.000000,16512.000000
mean,5.435235,1.096685,3.096961,1426.453004
std,2.387375,0.433215,11.578744,1137.056380
min,0.888889,0.333333,0.692308,3.000000
25%,4.452055,1.006508,2.428799,789.000000
50%,5.235874,1.049286,2.817240,1167.000000
75%,6.061037,1.100348,3.280000,1726.000000
max,141.909091,25.636364,1243.333333,35682.000000


In [6]:
def get_iqr_bounds(series, multiplier=1.5):
    Q1 = series.quantile(0.25)
    Q3 = series.quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - multiplier * IQR
    upper = Q3 + multiplier * IQR
    return lower, upper

for col in ['AveRooms', 'AveBedrms', 'AveOccup', 'Population']:
    lower, upper = get_iqr_bounds(X_train[col])
    n_outliers = ((X_train[col] < lower) | (X_train[col] > upper)).sum()
    print(f'{col}: bounds=({lower:.2f}, {upper:.2f}), outliers={n_outliers} ({n_outliers/len(X_train)*100:.1f}%)')


AveRooms: bounds=(2.04, 8.47), outliers=410 (2.5%)
AveBedrms: bounds=(0.87, 1.24), outliers=1153 (7.0%)
AveOccup: bounds=(1.15, 4.56), outliers=582 (3.5%)
Population: bounds=(-616.50, 3131.50), outliers=955 (5.8%)


In [7]:
def cap_outliers(X, columns, multiplier=1.5):
    X = X.copy()
    for col in columns:
        lower, upper = get_iqr_bounds(X[col], multiplier)
        X[col] = X[col].clip(lower, upper)
    return X

cols_to_check = ['AveRooms', 'AveBedrms', 'AveOccup', 'Population']
X_train_capped = cap_outliers(X_train, cols_to_check)
X_test_capped = cap_outliers(X_test, cols_to_check)

In [8]:
# Scale the features to prevent featrues with large numbers from unfairly dominating calculations
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_capped)
X_test_scaled = scaler.transform(X_test_capped)

In [11]:
# Find the best number of neighbors for the KNN regression model using cross-validation
param_grid = {'n_neighbors': range(1, 31, 2), 'weights': ['uniform', 'distance']}
grid = GridSearchCV(KNeighborsRegressor(), param_grid, cv=5, scoring='neg_mean_absolute_error')
grid.fit(X_train_scaled, y_train)
print(grid.best_params_)

{'n_neighbors': 11, 'weights': 'distance'}


In [12]:
# Fit model
knn_regressor = KNeighborsRegressor(**grid.best_params_)
knn_regressor.fit(X_train_scaled, y_train)
y_pred = knn_regressor.predict(X_test_scaled)
print(f'MSE: {mean_squared_error(y_test, y_pred)}')
print(f'R-squared score: {r2_score(y_test, y_pred)}')


MSE: 0.3677801352974048
R-squared score: 0.7193394265421511
